# 🧬 Algoritmo Genético — Problema do Caixeiro Viajante (TSP)

Este notebook implementa um **Algoritmo Genético (AG)** para resolver o **Problema do Caixeiro Viajante** com 20 cidades em um plano 1×1.

**Operadores utilizados:**
- Seleção por **Roleta Viciada**
- Crossover **Cíclico (CX)**
- Mutação por **Troca (Swap)**
- Elitismo de **50%**

## 1. Imports e Configuração

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

%matplotlib inline

# Parâmetros do algoritmo
POP_SIZE   = 20
GENERATIONS = 10000

print(f"Populacao: {POP_SIZE} individuos")
print(f"Geracoes : {GENERATIONS}")

## 2. Carregamento das Cidades

Tenta carregar `cidades.mat`. Se o arquivo não for encontrado, gera 20 cidades aleatórias com `seed=42`.

In [ ]:
try:
    data = np.loadtxt("cidades.mat")
    x, y = data[0], data[1]
    print("Cidades carregadas de cidades.mat")
except FileNotFoundError:
    np.random.seed(42)
    x = np.random.rand(20)
    y = np.random.rand(20)
    print("Arquivo nao encontrado — usando coordenadas aleatorias (seed=42)")

N_CIDADES = len(x)
print(f"Numero de cidades: {N_CIDADES}")

# Visualização das cidades
plt.figure(figsize=(6, 6))
plt.scatter(x, y, color='tomato', s=80, zorder=3)
for i in range(N_CIDADES):
    plt.annotate(str(i + 1), (x[i], y[i]), textcoords='offset points',
                 xytext=(6, 6), fontsize=8)
plt.title("Distribuição das 20 Cidades")
plt.xlabel("X")
plt.ylabel("Y")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 3. Classe `AlgoritmoGenetico`

Toda a lógica do AG está encapsulada nesta classe.

In [ ]:
class AlgoritmoGenetico:
    """Algoritmo Genetico para o Problema do Caixeiro Viajante (TSP).

    Cromossomos de permutacao (cidades 1 a n), selecao por roleta,
    crossover ciclico (CX), mutacao por troca e elitismo de 50%.
    """

    def __init__(self, x, y, tam_pop=20, geracoes=10000):
        self.x = x
        self.y = y
        self.n_cidades = len(x)
        self.tam_pop = tam_pop
        self.geracoes = geracoes
        self.matriz_custo = self._gerar_matriz_custo()
        self.populacao, self.aptidoes = self._iniciar_populacao()
        # Historico inclui o melhor da populacao inicial (geracao 0)
        self.historico = [self.aptidoes.min()]

    def _gerar_matriz_custo(self):
        """Gera matriz de distancias euclidiana usando broadcast do NumPy.
        Resultado: matriz n x n onde custo[i][j] = distancia entre cidade i e j.
        """
        coords = np.column_stack((self.x, self.y))
        diff = coords[:, np.newaxis, :] - coords[np.newaxis, :, :]
        return np.sqrt(np.sum(diff ** 2, axis=-1))

    def _iniciar_populacao(self):
        """Cria populacao inicial com permutacoes aleatorias de 1 a n_cidades."""
        pop = np.array(
            [np.random.permutation(self.n_cidades) + 1
             for _ in range(self.tam_pop)]
        )
        apt = np.array(
            [self._calcular_aptidao(pop[i]) for i in range(self.tam_pop)]
        )
        return pop, apt

    def _calcular_aptidao(self, config):
        """Distancia total do percurso (fitness). Ciclo fechado: retorna a cidade inicial."""
        idx = config - 1
        ciclo = np.append(idx, idx[0])
        return self.matriz_custo[ciclo[:-1], ciclo[1:]].sum()

    def _selecao_roleta(self, aptidoes, qtde):
        """Roleta: probabilidade inversamente proporcional a aptidao.
        replace=False garante que os dois pais sejam diferentes entre si.
        """
        pesos = 1.0 / aptidoes
        probs = pesos / pesos.sum()
        return np.random.choice(len(aptidoes), size=qtde, replace=False, p=probs)

    def _crossover_ciclo(self, p1, p2):
        """Crossover Ciclico (CX) conforme os 4 passos do trabalho.
        Passo 1: Troca valores em posicao aleatoria entre os pais.
        Passo 2: Localiza a duplicata no filho 1 e troca com filho 2.
        Passos 3+: Repete ate nao existirem mais duplicatas.
        """
        n = len(p1)
        f1, f2 = p1.copy(), p2.copy()
        idx = np.random.randint(n)
        if f1[idx] == f2[idx]:
            return f1, f2
        f1[idx], f2[idx] = f2[idx], f1[idx]
        while True:
            duplicatas = np.where(f1 == f1[idx])[0]
            if len(duplicatas) < 2:
                break
            dup_idx = duplicatas[0] if duplicatas[0] != idx else duplicatas[1]
            f1[dup_idx], f2[dup_idx] = f2[dup_idx], f1[dup_idx]
            idx = dup_idx
        return f1, f2

    def _mutacao(self, config):
        """Troca duas posicoes aleatorias no cromossomo (swap mutation)."""
        i, j = np.random.choice(self.n_cidades, 2, replace=False)
        config[i], config[j] = config[j], config[i]
        return config

    def evoluir(self):
        """Executa uma geracao completa do algoritmo genetico."""
        # Ordena por aptidao: menor distancia (mais apto) primeiro
        ordem = np.argsort(self.aptidoes)
        self.populacao = self.populacao[ordem]
        self.aptidoes = self.aptidoes[ordem]

        metade = self.tam_pop // 2
        nova_pop = np.empty_like(self.populacao)
        novas_apt = np.empty_like(self.aptidoes)

        # Elitismo: preserva os 50% melhores
        nova_pop[:metade] = self.populacao[:metade]
        novas_apt[:metade] = self.aptidoes[:metade]

        # Selecao por roleta opera apenas sobre a elite
        elite_apt = self.aptidoes[:metade]

        # Gera filhos ate completar a populacao
        idx_filho = metade
        while idx_filho < self.tam_pop:
            pais_idx = self._selecao_roleta(elite_apt, 2)
            p1 = self.populacao[pais_idx[0]]
            p2 = self.populacao[pais_idx[1]]
            f1, f2 = self._crossover_ciclo(p1, p2)
            for filho in (f1, f2):
                if idx_filho < self.tam_pop:
                    filho = self._mutacao(filho)
                    nova_pop[idx_filho] = filho
                    novas_apt[idx_filho] = self._calcular_aptidao(filho)
                    idx_filho += 1

        self.populacao = nova_pop
        self.aptidoes = novas_apt
        # Registra o melhor da geracao completa (inclui filhos)
        self.historico.append(self.aptidoes.min())

    def melhor_individuo(self):
        """Retorna (configuracao, aptidao) do melhor individuo da populacao atual."""
        idx = np.argmin(self.aptidoes)
        return self.populacao[idx], self.aptidoes[idx]

print("Classe AlgoritmoGenetico definida com sucesso.")

## 4. Inicialização e População Inicial

Cria o algoritmo e inspeciona o estado inicial antes de evoluir.

In [ ]:
ag = AlgoritmoGenetico(x, y, tam_pop=POP_SIZE, geracoes=GENERATIONS)

# Salva cópia da população inicial para exibir no relatório
pop_inicial = ag.populacao.copy()
apt_inicial = ag.aptidoes.copy()

separador = "-" * 55
print(separador)
print(f"  Tamanho da Populacao : {POP_SIZE}")
print(f"  Numero de Cidades    : {N_CIDADES}")
print(f"  Geracoes             : {GENERATIONS}")
print(separador)
print("\nPopulacao Inicial:")
for i in range(POP_SIZE):
    print(f"  {i+1:2d}. {pop_inicial[i].tolist()}  |  Custo: {apt_inicial[i]:.4f}")
print(f"\nMelhor custo inicial : {apt_inicial.min():.4f}")

## 5. Execução — 10.000 Gerações

> ⏱️ Pode levar alguns segundos dependendo do hardware.

In [ ]:
for geracao in range(GENERATIONS):
    ag.evoluir()

print(f"Evolucao concluida. {GENERATIONS} geracoes executadas.")
print(f"Melhor custo final: {ag.aptidoes.min():.4f}")

## 6. Relatório Final

In [ ]:
# Ordena população final para exibição
ordem = np.argsort(ag.aptidoes)
ag.populacao = ag.populacao[ordem]
ag.aptidoes = ag.aptidoes[ordem]
melhor_config, melhor_custo = ag.melhor_individuo()

print("\nPopulacao Final (ordenada por aptidao):")
for i in range(POP_SIZE):
    print(f"  {i+1:2d}. {ag.populacao[i].tolist()}  |  Custo: {ag.aptidoes[i]:.4f}")

print(f"\n{separador}")
print(f"  Melhor Custo   : {melhor_custo:.4f}")
print(f"  Melhor Solucao : {melhor_config.tolist()}")
print(separador)

## 7. Gráficos

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))

# --- Gráfico de convergência ---
ax1.plot(ag.historico, linewidth=0.8, color='steelblue')
ax1.set_title("Convergencia do Algoritmo Genetico")
ax1.set_xlabel("Geracao")
ax1.set_ylabel("Melhor Custo")
ax1.set_xlim(0, GENERATIONS)
ax1.grid(True, alpha=0.4)

# --- Gráfico da melhor rota ---
rota = melhor_config - 1
x_plot = np.append(x[rota], x[rota[0]])
y_plot = np.append(y[rota], y[rota[0]])

ax2.plot(x_plot, y_plot, '-', color='steelblue', linewidth=1.5, zorder=1)
ax2.scatter(x[rota], y[rota], color='tomato', s=60, zorder=3)

for i, cidade in enumerate(rota):
    ax2.annotate(
        str(cidade + 1),
        (x[cidade], y[cidade]),
        textcoords='offset points',
        xytext=(6, 6),
        fontsize=8
    )

ax2.plot(x_plot[0], y_plot[0], 's', color='gold',
         markersize=10, zorder=4, label=f'Inicio (Cidade {rota[0]+1})')
ax2.set_title(f"Melhor Caminho Encontrado\nCusto: {melhor_custo:.4f}")
ax2.set_xlabel("Coordenada X")
ax2.set_ylabel("Coordenada Y")
ax2.legend()
ax2.grid(True, alpha=0.4)

plt.tight_layout()
plt.savefig("resultado.png", dpi=150, bbox_inches='tight')
plt.show()
print("Grafico salvo em resultado.png")

## 8. Experimento: Inspecionando o Crossover Cíclico

Célula extra para entender o CX com o exemplo do enunciado.

In [ ]:
# Reproduz o exemplo da Figura 1 do enunciado
p1 = np.array([4, 1, 5, 3, 2, 6])
p2 = np.array([3, 4, 6, 2, 1, 5])

# Força a posição 3 para reproduzir o exemplo
ag_teste = AlgoritmoGenetico(x[:6], y[:6], tam_pop=2, geracoes=1)

print("Exemplo do enunciado (posicao 3):")
print(f"  Pai 1 : {p1.tolist()}")
print(f"  Pai 2 : {p2.tolist()}")

np.random.seed(0)  # para reproducibilidade
f1, f2 = ag_teste._crossover_ciclo(p1, p2)
print(f"  Filho 1: {f1.tolist()}")
print(f"  Filho 2: {f2.tolist()}")
print(f"  Esperado: [3, 4, 5, 2, 1, 6] e [4, 1, 6, 3, 2, 5]")
print(f"  Sem duplicatas f1: {len(set(f1)) == len(f1)}")
print(f"  Sem duplicatas f2: {len(set(f2)) == len(f2)}")

## 9. Experimento: Convergência em Menos Gerações

Útil para explorar o comportamento do algoritmo com menos iterações.

In [ ]:
# Testa com 1000 gerações para ver como o algoritmo evolui mais cedo
ag_rapido = AlgoritmoGenetico(x, y, tam_pop=POP_SIZE, geracoes=1000)
for _ in range(1000):
    ag_rapido.evoluir()

_, custo_1000 = ag_rapido.melhor_individuo()

plt.figure(figsize=(8, 4))
plt.plot(ag_rapido.historico, color='steelblue', linewidth=1)
plt.title(f"Convergencia em 1000 geracoes — Melhor custo: {custo_1000:.4f}")
plt.xlabel("Geracao")
plt.ylabel("Melhor Custo")
plt.grid(True, alpha=0.4)
plt.tight_layout()
plt.show()

print(f"Custo com 1000 geracoes : {custo_1000:.4f}")
print(f"Custo com 10000 geracoes: {melhor_custo:.4f}")
print(f"Melhoria adicional      : {((custo_1000 - melhor_custo) / custo_1000 * 100):.1f}%")